# Prepare FineWeb-Edu Dataset

This notebook downloads and tokenizes ~2B tokens from FineWeb-Edu for training a 57M parameter model.

**Splits:**
- Train: 95% (~1.9B tokens)
- Val: 2.5% (~50M tokens) 
- Test: 2.5% (~50M tokens)

In [ ]:
# Install dependencies if needed
# !pip install datasets transformers tqdm

In [ ]:
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm.auto import tqdm

# Configuration
TOKENIZER_NAME = "gpt2"  # Or path to your custom tokenizer
OUTPUT_DIR = Path("../data/fineweb-edu")
TARGET_TOKENS = 2_000_000_000  # 2B tokens
SEED = 42

## 1. Load Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
print(f"Tokenizer vocab size: {len(tokenizer)}")
print(f"EOS token: {tokenizer.eos_token} (id={tokenizer.eos_token_id})")

## 2. Download FineWeb-Edu (streaming)

We use streaming to avoid downloading the entire dataset (~1TB+). We'll stop once we have enough tokens.

In [ ]:
# Stream FineWeb-Edu
dataset = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    split="train",
    streaming=True,
)

print("Streaming dataset loaded")
print(f"Features: {dataset.features}")

In [ ]:
# Collect documents until we reach target tokens
documents = []
total_tokens = 0

pbar = tqdm(total=TARGET_TOKENS, unit="tok", desc="Collecting tokens")

for i, example in enumerate(dataset):
    # Tokenize
    tokens = tokenizer.encode(example["text"], add_special_tokens=False)
    
    # Skip very short documents (< 50 tokens)
    if len(tokens) < 50:
        continue
    
    documents.append({
        "input_ids": tokens,
        "uid": len(documents),
    })
    
    total_tokens += len(tokens)
    pbar.update(len(tokens))
    
    if total_tokens >= TARGET_TOKENS:
        break
    
    # Progress update every 10k docs
    if len(documents) % 10000 == 0:
        pbar.set_postfix({"docs": len(documents), "avg_len": total_tokens // len(documents)})

pbar.close()

print(f"\nCollected {len(documents):,} documents")
print(f"Total tokens: {total_tokens:,} ({total_tokens/1e9:.2f}B)")
print(f"Average doc length: {total_tokens // len(documents)} tokens")

## 3. Create Train/Val/Test Splits

Standard split ratios:
- **Train**: 95% - Used for gradient updates
- **Val**: 2.5% - Used during training to monitor overfitting
- **Test**: 2.5% - Held out, only used for final evaluation

In [ ]:
import random
from datasets import Dataset

# Shuffle documents
random.seed(SEED)
random.shuffle(documents)

# Split ratios
n = len(documents)
train_end = int(0.95 * n)
val_end = int(0.975 * n)

train_docs = documents[:train_end]
val_docs = documents[train_end:val_end]
test_docs = documents[val_end:]

print(f"Train: {len(train_docs):,} documents")
print(f"Val:   {len(val_docs):,} documents")
print(f"Test:  {len(test_docs):,} documents")

In [ ]:
# Count tokens per split
train_tokens = sum(len(d["input_ids"]) for d in train_docs)
val_tokens = sum(len(d["input_ids"]) for d in val_docs)
test_tokens = sum(len(d["input_ids"]) for d in test_docs)

print(f"Train tokens: {train_tokens:,} ({train_tokens/1e9:.2f}B)")
print(f"Val tokens:   {val_tokens:,} ({val_tokens/1e6:.0f}M)")
print(f"Test tokens:  {test_tokens:,} ({test_tokens/1e6:.0f}M)")

## 4. Convert to HuggingFace Datasets and Save

In [ ]:
# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Convert to HF datasets
train_dataset = Dataset.from_list(train_docs)
val_dataset = Dataset.from_list(val_docs)
test_dataset = Dataset.from_list(test_docs)

print(f"Train dataset: {train_dataset}")
print(f"Val dataset:   {val_dataset}")
print(f"Test dataset:  {test_dataset}")

In [ ]:
# Save to disk
train_dataset.save_to_disk(OUTPUT_DIR / "train")
val_dataset.save_to_disk(OUTPUT_DIR / "val")
test_dataset.save_to_disk(OUTPUT_DIR / "test")

print(f"\nSaved to {OUTPUT_DIR}")
print(f"  train/: {len(train_dataset):,} documents")
print(f"  val/:   {len(val_dataset):,} documents")
print(f"  test/:  {len(test_dataset):,} documents")

## 5. Verify Data

In [ ]:
from datasets import load_from_disk

# Reload and verify
train_check = load_from_disk(str(OUTPUT_DIR / "train"))

print(f"Loaded train dataset: {len(train_check)} documents")
print(f"Columns: {train_check.column_names}")
print(f"\nFirst document preview:")
print(f"  UID: {train_check[0]['uid']}")
print(f"  Length: {len(train_check[0]['input_ids'])} tokens")
print(f"  First 20 tokens: {train_check[0]['input_ids'][:20]}")
print(f"  Decoded: {tokenizer.decode(train_check[0]['input_ids'][:50])}...")

## 6. Update Config

Now update your `config.yaml`:

```yaml
paths:
  tokenizer: gpt2
  train_data: ./data/fineweb-edu/train
  val_data: ./data/fineweb-edu/val
  test_data: ./data/fineweb-edu/test
  output_dir: ./outputs
```

Then run training:
```bash
python -m src.train config.yaml
```

In [ ]:
# Summary
print("="*50)
print("DATA PREPARATION COMPLETE")
print("="*50)
print(f"Location: {OUTPUT_DIR.absolute()}")
print(f"Tokenizer: {TOKENIZER_NAME}")
print(f"Vocab size: {len(tokenizer)}")
print(f"")
print(f"Train: {len(train_docs):,} docs, {train_tokens/1e9:.2f}B tokens")
print(f"Val:   {len(val_docs):,} docs, {val_tokens/1e6:.0f}M tokens")
print(f"Test:  {len(test_docs):,} docs, {test_tokens/1e6:.0f}M tokens")
print(f"")
print(f"Total: {total_tokens/1e9:.2f}B tokens")